# Equity Research: DCF Valuation PT Unilever Indonesia Tbk (UNVR)

**Tujuan:** Mengestimasi intrinsic value saham UNVR menggunakan metode
Discounted Cash Flow (DCF) dan membandingkannya dengan harga pasar.

**Metode:** Free Cash Flow to Firm (FCFF) DCF dengan terminal value

**Data:** Yahoo Finance (via yfinance) + laporan keuangan UNVR (IDX)

**Disclaimer:** Analisis ini dibuat untuk tujuan edukasi dan portofolio.
Bukan merupakan rekomendasi investasi.

---

## 0. Setup

In [ ]:
# Install yfinance kalau belum ada
# !pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)

TICKER = 'UNVR.JK'   # kode UNVR di Yahoo Finance (Jakarta)
print(f'Setup selesai! Ticker: {TICKER}')

## 1. Pull data dari Yahoo Finance

In [ ]:
# Download data UNVR
unvr = yf.Ticker(TICKER)

# Info perusahaan
info = unvr.info
print('=== PROFIL PERUSAHAAN ===')
print(f'Nama         : {info.get("longName", "N/A")}')
print(f'Sektor       : {info.get("sector", "N/A")}')
print(f'Industri     : {info.get("industry", "N/A")}')
print(f'Harga terkini: Rp {info.get("currentPrice", "N/A"):,}')
print(f'Market Cap   : Rp {info.get("marketCap", 0)/1e12:.2f} Triliun')
print(f'Shares Out   : {info.get("sharesOutstanding", 0)/1e9:.2f} Miliar lembar')

In [ ]:
# Harga saham historis 5 tahun
hist = unvr.history(period='5y')
print(f'Data harga: {len(hist)} hari trading')
print(f'Periode   : {hist.index[0].strftime("%b %Y")} – {hist.index[-1].strftime("%b %Y")}')
hist[['Close', 'Volume']].tail()

In [ ]:
# Laporan keuangan
income_stmt = unvr.financials          # Income Statement
balance_sheet = unvr.balance_sheet     # Balance Sheet
cashflow = unvr.cashflow               # Cash Flow Statement

print('Income Statement — baris tersedia:')
print(list(income_stmt.index))
print()
print('Cash Flow — baris tersedia:')
print(list(cashflow.index))

## 2. Visualisasi harga saham historis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('UNVR.JK — Analisis Harga Saham 5 Tahun', fontsize=15, fontweight='bold')

# Plot harga
axes[0].plot(hist.index, hist['Close'], color='steelblue', linewidth=1.5)
axes[0].fill_between(hist.index, hist['Close'], alpha=0.1, color='steelblue')

# Moving averages
ma50  = hist['Close'].rolling(50).mean()
ma200 = hist['Close'].rolling(200).mean()
axes[0].plot(hist.index, ma50,  color='orange', linewidth=1.2, linestyle='--', label='MA-50')
axes[0].plot(hist.index, ma200, color='red',    linewidth=1.2, linestyle='--', label='MA-200')

axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
axes[0].set_title('Harga Penutupan + Moving Average')
axes[0].set_ylabel('Harga (Rp)')
axes[0].legend()

# Volume
axes[1].bar(hist.index, hist['Volume']/1e6, color='steelblue', alpha=0.6, width=1)
axes[1].set_title('Volume Perdagangan')
axes[1].set_ylabel('Volume (Juta lembar)')
axes[1].set_xlabel('Tanggal')

plt.tight_layout()
plt.savefig('../outputs/figures/04_unvr_harga_historis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ekstrak data keuangan historis

Kita ambil data 4 tahun terakhir untuk menghitung rata-rata dan tren.

In [ ]:
# Fungsi helper ambil data dari finansial
def get_item(df, keywords):
    """Ambil baris dari dataframe berdasarkan keyword"""
    for kw in keywords:
        matches = [i for i in df.index if kw.lower() in i.lower()]
        if matches:
            return df.loc[matches[0]]
    return pd.Series([np.nan]*len(df.columns), index=df.columns)

# Ekstrak komponen penting (dalam Rupiah)
revenue   = get_item(income_stmt, ['Total Revenue', 'Revenue'])
ebit      = get_item(income_stmt, ['EBIT', 'Operating Income'])
net_income= get_item(income_stmt, ['Net Income'])
tax_rate_raw = get_item(income_stmt, ['Tax', 'Income Tax'])
depreciation = get_item(cashflow, ['Depreciation', 'Depreciation And Amortization'])
capex     = get_item(cashflow, ['Capital Expenditure', 'Capex'])
delta_wc  = get_item(cashflow, ['Change In Working Capital', 'Working Capital'])

# Susun tabel ringkasan
fin_summary = pd.DataFrame({
    'Revenue (Rp M)':      revenue / 1e6,
    'EBIT (Rp M)':         ebit / 1e6,
    'Net Income (Rp M)':   net_income / 1e6,
    'Depreciation (Rp M)': depreciation / 1e6,
    'CapEx (Rp M)':        capex / 1e6,
}).T

fin_summary.columns = [str(c)[:4] for c in fin_summary.columns]
print('=== RINGKASAN KEUANGAN UNVR (Rp Miliar) ===')
print(fin_summary.round(0))

In [ ]:
# Visualisasi tren keuangan
years = [str(c)[:4] for c in income_stmt.columns]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Tren Keuangan UNVR', fontsize=14, fontweight='bold')

rev_vals = revenue.values / 1e12
ni_vals  = net_income.values / 1e12
ebit_vals = ebit.values / 1e12

axes[0].bar(years, rev_vals, color='steelblue', alpha=0.8)
axes[0].set_title('Revenue (Rp Triliun)')
axes[0].set_ylabel('Rp Triliun')
for i, v in enumerate(rev_vals):
    if not np.isnan(v):
        axes[0].text(i, v + 0.2, f'{v:.1f}T', ha='center', fontsize=9)

axes[1].bar(years, ebit_vals, color='seagreen', alpha=0.8)
axes[1].set_title('EBIT (Rp Triliun)')
for i, v in enumerate(ebit_vals):
    if not np.isnan(v):
        axes[1].text(i, v + 0.1, f'{v:.1f}T', ha='center', fontsize=9)

axes[2].bar(years, ni_vals, color='coral', alpha=0.8)
axes[2].set_title('Net Income (Rp Triliun)')
for i, v in enumerate(ni_vals):
    if not np.isnan(v):
        axes[2].text(i, v + 0.05, f'{v:.1f}T', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/04_unvr_tren_keuangan.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Hitung Free Cash Flow to Firm (FCFF)

**Formula:**
$$FCFF = EBIT \times (1 - Tax\ Rate) + Depreciation - CapEx - \Delta Working\ Capital$$

In [ ]:
# Hitung tax rate rata-rata
try:
    tax_expense = get_item(income_stmt, ['Tax', 'Income Tax'])
    pretax      = get_item(income_stmt, ['Pretax Income', 'Income Before Tax'])
    tax_rates   = (tax_expense / pretax).abs()
    effective_tax_rate = tax_rates.mean()
except:
    effective_tax_rate = 0.22  # fallback: tarif PPh Badan Indonesia

print(f'Effective tax rate: {effective_tax_rate:.2%}')

# Hitung FCFF tiap tahun
nopat = ebit * (1 - effective_tax_rate)      # Net Operating Profit After Tax
fcff  = nopat + depreciation - abs(capex) - abs(delta_wc)

fcff_df = pd.DataFrame({
    'EBIT (Rp M)':        ebit / 1e6,
    'NOPAT (Rp M)':       nopat / 1e6,
    'Depreciation (Rp M)':depreciation / 1e6,
    'CapEx (Rp M)':       capex / 1e6,
    'FCFF (Rp M)':        fcff / 1e6,
}).T
fcff_df.columns = years

print('\n=== FCFF UNVR (Rp Miliar) ===')
print(fcff_df.round(0))

## 5. Asumsi DCF

Ini adalah bagian paling kritis — asumsi menentukan hasil valuasi.
Kita buat tiga skenario: **Bull, Base, Bear**.

In [ ]:
# === ASUMSI DCF ===
# Ubah nilai-nilai ini sesuai analisis kamu

# WACC — Weighted Average Cost of Capital
# Estimasi: risk-free rate (SBN 10Y ~6.5%) + equity risk premium + beta
WACC = 0.11   # 11% — konservatif untuk FMCG Indonesia

# Terminal growth rate (pertumbuhan jangka panjang)
# Tidak boleh melebihi pertumbuhan GDP jangka panjang Indonesia (~5%)
TERMINAL_GROWTH = 0.04   # 4%

# Proyeksi tahun
PROJECTION_YEARS = 5

# Skenario growth rate FCFF (5 tahun proyeksi)
scenarios = {
    'Bull':  0.08,   # 8%  — recovery kuat, ekspansi produk
    'Base':  0.05,   # 5%  — pertumbuhan stabil sesuai GDP
    'Bear': -0.02,   # -2% — tekanan kompetisi, penurunan margin
}

# FCFF base (rata-rata 2 tahun terakhir)
fcff_base = fcff.iloc[:2].mean()
shares_outstanding = info.get('sharesOutstanding', 38_150_000_000)
current_price = info.get('currentPrice', 0)

print('=== ASUMSI DCF ===')
print(f'WACC               : {WACC:.1%}')
print(f'Terminal Growth    : {TERMINAL_GROWTH:.1%}')
print(f'Projection Years   : {PROJECTION_YEARS} tahun')
print(f'FCFF Base (Rp M)   : {fcff_base/1e6:,.0f}')
print(f'Shares Outstanding : {shares_outstanding/1e9:.2f} Miliar')
print(f'Harga Pasar Saat ini: Rp {current_price:,}')

## 6. Kalkulasi DCF Valuation

In [ ]:
def dcf_valuation(fcff_0, growth_rate, wacc, terminal_growth, years, shares):
    """Hitung intrinsic value per saham dengan DCF."""
    # Proyeksi FCFF
    projected_fcff = []
    for t in range(1, years + 1):
        fcff_t = fcff_0 * (1 + growth_rate) ** t
        projected_fcff.append(fcff_t)
    
    # Present value of projected FCFF
    pv_fcff = sum(
        fcff_t / (1 + wacc) ** t
        for t, fcff_t in enumerate(projected_fcff, 1)
    )
    
    # Terminal value (Gordon Growth Model)
    fcff_terminal = projected_fcff[-1] * (1 + terminal_growth)
    terminal_value = fcff_terminal / (wacc - terminal_growth)
    pv_terminal    = terminal_value / (1 + wacc) ** years
    
    # Enterprise value & equity value
    enterprise_value = pv_fcff + pv_terminal
    
    # Net debt (untuk konversi EV → Equity Value)
    try:
        total_debt = balance_sheet.loc[
            [i for i in balance_sheet.index if 'debt' in i.lower()][0]
        ].iloc[0]
        cash = balance_sheet.loc[
            [i for i in balance_sheet.index if 'cash' in i.lower()][0]
        ].iloc[0]
        net_debt = total_debt - cash
    except:
        net_debt = 0
    
    equity_value     = enterprise_value - net_debt
    intrinsic_value  = equity_value / shares
    
    return {
        'projected_fcff':   projected_fcff,
        'pv_fcff':          pv_fcff,
        'terminal_value':   terminal_value,
        'pv_terminal':      pv_terminal,
        'enterprise_value': enterprise_value,
        'equity_value':     equity_value,
        'intrinsic_value':  intrinsic_value,
    }

# Jalankan tiga skenario
results = {}
for scenario, growth in scenarios.items():
    results[scenario] = dcf_valuation(
        fcff_base, growth, WACC, TERMINAL_GROWTH,
        PROJECTION_YEARS, shares_outstanding
    )

# Tampilkan hasil
print('=== HASIL DCF VALUATION UNVR ===')
print(f'{"":30} {"Bull":>12} {"Base":>12} {"Bear":>12}')
print('-' * 70)
print(f'{"Growth Rate":30} {scenarios["Bull"]:>12.1%} {scenarios["Base"]:>12.1%} {scenarios["Bear"]:>12.1%}')
print(f'{"PV FCFF (Rp M)":30} {results["Bull"]["pv_fcff"]/1e6:>12,.0f} {results["Base"]["pv_fcff"]/1e6:>12,.0f} {results["Bear"]["pv_fcff"]/1e6:>12,.0f}')
print(f'{"Terminal Value (Rp M)":30} {results["Bull"]["terminal_value"]/1e6:>12,.0f} {results["Base"]["terminal_value"]/1e6:>12,.0f} {results["Bear"]["terminal_value"]/1e6:>12,.0f}')
print(f'{"Enterprise Value (Rp M)":30} {results["Bull"]["enterprise_value"]/1e6:>12,.0f} {results["Base"]["enterprise_value"]/1e6:>12,.0f} {results["Bear"]["enterprise_value"]/1e6:>12,.0f}')
print(f'{"Intrinsic Value / Share":30} {results["Bull"]["intrinsic_value"]:>12,.0f} {results["Base"]["intrinsic_value"]:>12,.0f} {results["Bear"]["intrinsic_value"]:>12,.0f}')
print(f'{"Harga Pasar":30} {current_price:>12,.0f} {current_price:>12,.0f} {current_price:>12,.0f}')
print()
for scenario in scenarios:
    iv = results[scenario]['intrinsic_value']
    updown = ((iv - current_price) / current_price * 100) if current_price else 0
    verdict = 'UNDERVALUED ✓' if iv > current_price else 'OVERVALUED ✗'
    print(f'{scenario:5}: Rp {iv:,.0f}/saham → {updown:+.1f}% vs pasar → {verdict}')

## 7. Visualisasi hasil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('DCF Valuation UNVR — Hasil Tiga Skenario', fontsize=14, fontweight='bold')

# Plot 1: Intrinsic value vs harga pasar
scenario_names  = list(scenarios.keys())
intrinsic_values = [results[s]['intrinsic_value'] for s in scenario_names]
colors = ['seagreen', 'steelblue', 'coral']

bars = axes[0].bar(scenario_names, intrinsic_values, color=colors, alpha=0.85, width=0.5)
axes[0].axhline(y=current_price, color='red', linestyle='--', linewidth=2,
                label=f'Harga Pasar: Rp {current_price:,}')
axes[0].set_title('Intrinsic Value per Saham vs Harga Pasar')
axes[0].set_ylabel('Harga (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'Rp {x:,.0f}'))
axes[0].legend()
for bar, val in zip(bars, intrinsic_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'Rp {val:,.0f}', ha='center', fontsize=10, fontweight='bold')

# Plot 2: Proyeksi FCFF Base scenario
proj_years = [f'Y{i}' for i in range(1, PROJECTION_YEARS+1)]
proj_fcff  = [f/1e12 for f in results['Base']['projected_fcff']]

axes[1].bar(proj_years, proj_fcff, color='steelblue', alpha=0.8)
axes[1].set_title(f'Proyeksi FCFF — Skenario Base (growth {scenarios["Base"]:.0%}/tahun)')
axes[1].set_ylabel('FCFF (Rp Triliun)')
axes[1].set_xlabel('Tahun Proyeksi')
for i, v in enumerate(proj_fcff):
    axes[1].text(i, v + 0.05, f'{v:.2f}T', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/04_unvr_dcf_valuation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sensitivity Analysis — WACC vs Growth Rate

In [ ]:
# Sensitivity table: intrinsic value pada berbagai kombinasi WACC & growth
wacc_range   = [0.09, 0.10, 0.11, 0.12, 0.13]
growth_range = [0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08]

sensitivity = pd.DataFrame(index=[f'{w:.0%}' for w in wacc_range],
                            columns=[f'{g:.0%}' for g in growth_range])

for w in wacc_range:
    for g in growth_range:
        res = dcf_valuation(fcff_base, g, w, TERMINAL_GROWTH,
                            PROJECTION_YEARS, shares_outstanding)
        sensitivity.loc[f'{w:.0%}', f'{g:.0%}'] = round(res['intrinsic_value'])

sensitivity = sensitivity.astype(float)

print(f'Sensitivity Analysis: Intrinsic Value per Saham (Rp)')
print(f'Baris = WACC | Kolom = Growth Rate FCFF')
print(f'Harga Pasar Saat Ini = Rp {current_price:,}')
print()
print(sensitivity.to_string())

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 6))

# Warna: hijau = undervalued, merah = overvalued vs harga pasar
sns.heatmap(sensitivity,
            annot=True, fmt=',.0f',
            cmap='RdYlGn', center=current_price,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Intrinsic Value (Rp/saham)'})

ax.set_xlabel('Growth Rate FCFF (%/tahun)', fontsize=12)
ax.set_ylabel('WACC', fontsize=12)
ax.set_title(f'Sensitivity Analysis DCF UNVR\nHijau = Undervalued vs Rp {current_price:,} | Merah = Overvalued',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/04_unvr_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Investment Thesis & Kesimpulan

**Isi bagian ini setelah melihat semua output di atas.**

**Pertanyaan panduan:**
1. Berapa intrinsic value UNVR di skenario Base? Undervalued atau overvalued?
2. Asumsi WACC dan growth rate yang kamu pakai — apakah reasonable? Kenapa?
3. Dari sensitivity analysis, pada kombinasi WACC & growth berapa UNVR terlihat menarik?
4. Apa risiko utama yang bisa membuat skenario Bull tidak terjadi?
5. Rekomendasi akhir: BUY / HOLD / SELL? Dengan target harga berapa?

**Investment Thesis:**

*[Tulis thesis investasimu di sini — 2-3 paragraf]*

**Risiko:**

*[Tulis risiko utama di sini]*

**Rekomendasi:**

| Skenario | Intrinsic Value | vs Pasar | Rekomendasi |
|----------|----------------|----------|-------------|
| Bull | Rp ? | ?% | ? |
| Base | Rp ? | ?% | ? |
| Bear | Rp ? | ?% | ? |

---
*Disclaimer: Analisis ini dibuat untuk tujuan edukasi. Bukan rekomendasi investasi.*

*Analisis oleh: Armandya Danu | Ekonomi Universitas Brawijaya | Mei 2026*